# Growing the freelancer segment
One notebook, end to end: market → funnel → attribution and incrementality → experiments → creatives and referral → allocation → plan.

Market numbers are **public** (URSSAF, INSEE). Channels, spend, sign-ups and tests are **simulated** (seed 42) and are not Alan data. Run `python run_all.py` first: this notebook reads the SQL and statistics outputs in `data/output/`.

In [1]:
import duckdb, pandas as pd
pd.set_option('display.width', 140); pd.set_option('display.precision', 3)
con = duckdb.connect()
q = lambda sql: con.sql(sql.replace('OUT/', '../data/output/')).df()
q("SELECT test, passed FROM 'OUT/tests.csv'").passed.value_counts()

passed
True    20
Name: count, dtype: int64

## 1 · Market (PUBLIC DATA)
Size on the economically active auto-entrepreneurs, not the registered ones.

In [2]:
q("SELECT ti_total, ae_eco_active_q4_2025, ae_registered_q4_2025, active_share, creations_per_month_12m FROM 'OUT/market_kpis.csv'")

,ti_total,ae_eco_active_q4_2025,ae_registered_q4_2025,active_share,creations_per_month_12m
0,3.630e+06,1.660e+06,3.031e+06,0.548,71817.583


In [3]:
q("SELECT departement, eco_active, round(income_ae) income_ae, round(density_km2, 1) density FROM 'OUT/market_dept.csv' WHERE target ORDER BY value_rank")

,departement,eco_active,income_ae,density
0,Paris,96879.0,10026.0,922.7
1,Rhône,54128.0,8509.0,16.6
2,Gironde,54654.0,8013.0,5.3
3,Bouches-du-Rhône,58446.0,7320.0,11.5
4,Hauts-de-Seine,41505.0,9860.0,237.2


In [4]:
q("SELECT mo, round(100 * (avg_factor - 1)) AS pct_vs_trend FROM 'OUT/insee_season_profile.csv'").T

,0,1,2,3,4,5,6,7,8,9,10,11
mo,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0
pct_vs_trend,11.0,3.0,12.0,1.0,-7.0,-0.0,-9.0,-20.0,10.0,17.0,-3.0,-13.0


## 2 · Funnel and Simpson's paradox (SIMULATED)

In [5]:
q("SELECT channel, round(spend) spend, signups_real, retention_3m, cost_per_real, conv_click FROM 'OUT/funnel_channel.csv'")

,channel,spend,signups_real,retention_3m,cost_per_real,conv_click
0,Brand search,1.007e+06,43621,0.785,23.096,0.071
1,Non-brand search,1.454e+06,32571,0.720,44.633,0.033
2,LinkedIn Ads,1.248e+06,22071,0.775,56.544,0.039
3,Partnerships,6.240e+05,13251,0.843,47.091,0.033
4,Referral,4.164e+05,9298,0.795,44.781,0.149
5,AI search,0.000e+00,3043,0.813,0.000,0.066


In [6]:
q("SELECT * FROM 'OUT/simpson.csv'").pivot(index='subsegment', columns='channel', values='conv').round(4)

channel,LinkedIn Ads,Non-brand search,Partnerships
subsegment,,,
All sub-segments,0.038,0.033,0.033
Commerce & services,0.021,0.026,0.024
Construction & crafts,0.025,0.030,0.029
Consulting & IT,0.058,0.070,0.063
Health & wellbeing,0.026,0.034,0.032
New founders (<6 months),0.032,0.039,0.038


## 3 · Attribution vs incrementality (E1, E2, X3)

In [7]:
q("SELECT * FROM 'OUT/waterfall.csv'")

,step,label,signups
0,1,Reported by platforms,173933.000
1,2,Deduplicated,123855.000
2,3,Incremental,48261.967


In [8]:
q("SELECT channel, incremental_share, source, round(cost_per_incremental) cpi, round(cost_per_reported) cpr FROM 'OUT/incremental_channel.csv'")

,channel,incremental_share,source,cpi,cpr
0,Brand search,0.013,measured (X4),1789.0,18.0
1,Non-brand search,0.600,assumed,74.0,33.0
2,LinkedIn Ads,0.376,measured (X3),150.0,29.0
3,Partnerships,0.800,assumed,59.0,34.0
4,Referral,0.700,assumed,64.0,45.0
5,AI search,0.900,assumed,0.0,0.0


## 4 · Experiments with 95% intervals (E3, E5, X3, X4)

In [9]:
q("SELECT label, effect, lo, hi, decision FROM 'OUT/experiment_results.csv'")

,label,effect,lo,hi,decision
0,X1 landing page by sub-segment,0.069,0.006,0.132,Invalid (sample-ratio mismatch): fix the bot f...
1,"X2 referral reward, weeks 1-3",0.404,0.323,0.485,Early lift (novelty)
2,"X2 referral reward, weeks 4-8",-0.005,-0.053,0.043,Novelty only: keep EUR 50
3,X3 LinkedIn Ads geo test (spend x3),0.058,-0.012,0.128,"Inconclusive: hold spend, retest longer"
4,X4 brand search holdout,-0.004,-0.052,0.047,Cut brand search to a defence budget


In [10]:
q("SELECT share_test, srm_chi2, decision FROM 'OUT/x1.csv'")

,share_test,srm_chi2,decision
0,0.517,120.122,"invalid: sample-ratio mismatch, fix the bot fi..."


## 5 · Creatives and referral (E6, E7)

In [11]:
q("SELECT * FROM 'OUT/creative_fatigue_rule.csv'")

,ctr_ratio_5_6_vs_1_4,first_freq_below_60pct
0,0.524,5


In [12]:
q("SELECT subsegment, angle, round(ctr_index, 2) ctr_index FROM 'OUT/creative_angle_seg.csv' WHERE winner")

,subsegment,angle,ctr_index
0,Commerce & services,Price,1.45
1,Construction & crafts,Fast reimbursement,1.35
2,Consulting & IT,App experience,1.40
3,Health & wellbeing,Prevention,1.37
4,New founders (<6 months),Made for freelancers,1.38


In [13]:
q("SELECT referrers, referees, flagged_codes, flagged_referees, p99_referees, active_rate_all, active_rate_clean FROM 'OUT/referral_summary.csv'")

,referrers,referees,flagged_codes,flagged_referees,p99_referees,active_rate_all,active_rate_clean
0,3973,9298,12,361,8.0,0.795,0.821


## 6 · The August alert (E8 / F1)
The naive month-over-month rule fires; once sign-ups are deseasonalised with the INSEE creation seasonality, it does not.

In [14]:
q("SELECT strftime(month, '%Y-%m') AS ym, round(mom_raw, 3) mom_raw, round(mom_deseason, 3) mom_deseason, naive_alert, deseason_alert FROM 'OUT/alerts.csv' WHERE naive_alert OR deseason_alert")

,ym,mom_raw,mom_deseason,naive_alert,deseason_alert
0,2025-02,-0.152,0.022,True,False
1,2025-05,-0.153,0.013,True,False
2,2025-08,-0.234,0.011,True,False
3,2025-11,-0.353,-0.034,True,False
4,2026-05,-0.267,-0.039,True,False
5,2026-08,-0.212,0.036,True,False
6,2026-11,-0.303,-0.002,True,False


## 7 · Where the next euro goes (response curves, same budget)

In [15]:
a = q("SELECT channel, spend_now, spend_opt, round(inc_now, 1) inc_now, round(inc_opt, 1) inc_opt, round(mcpa_now) mcpa_now, round(mcpa_opt) mcpa_opt FROM 'OUT/allocation.csv'")
print(f"moved {a.eval('spend_opt - spend_now').clip(lower=0).sum() / a.spend_now.sum():.0%} of the budget, incremental {a.inc_opt.sum() / a.inc_now.sum() - 1:+.0%}")
a

moved 19% of the budget, incremental +11%


,channel,spend_now,spend_opt,inc_now,inc_opt,mcpa_now,mcpa_opt
0,Brand search,10000,2900.0,48.0,30.0,1250.0,145.0
1,Non-brand search,14000,12600.0,189.9,182.0,188.0,166.0
2,LinkedIn Ads,12000,13600.0,109.9,121.5,135.0,142.0
3,Partnerships,6000,12000.0,95.1,154.5,82.0,124.0
4,Referral,4000,5000.0,60.0,69.2,100.0,117.0


## 8 · Plan
See `docs/plan_90_days.md`: reallocate in month 1 (brand search to defence level, partnerships doubled, referral guardrails, creative rotation at frequency 4), measure in months 2-3 (LinkedIn geo retest, X1 rerun, AI search panel), review every week.